
# Tutorial 3: Adam with a $\rho$-Scaled Learning Rate

This notebook is the Adam counterpart to Tutorial 0.

There, we directly capped an SGD step using a batch JVP estimate of $\rho$.
Here, we use the packaged `RhoScaledAdam` optimizer from the library. It monitors
$\rho$ from the current logits and shrinks Adam's global learning-rate scale when
the model approaches instability.

In this notebook we compare:
- fixed Adam with three learning rates,
- rho-scaled Adam with the same three learning rates.

We again use the `sklearn` digits dataset so the notebook stays fast and interactive.


In [ ]:

from pathlib import Path
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "tutorials").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repo root: {REPO}")



## 1. Setup

The structure mirrors the SGD tutorial:
- one architecture (`MLP`),
- one seed,
- three Adam learning rates spanning safe, aggressive, and unstable behavior,
- one optional multiseed block at the end.

The goal is to see how the rho-scaled optimizer changes Adam's behavior without
drowning the reader in architecture or hyperparameter details.


In [ ]:

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from ghosts.control import RhoScaledAdam, computeRho

torch.set_num_threads(1)

DEVICE = torch.device("cpu")
LOW_LR = 0.001
MID_LR = 0.05
HIGH_LR = 0.5
LRS = [LOW_LR, MID_LR, HIGH_LR]
EPOCHS = 24
BATCH_SIZE = 128
RHO_CAP = 1.0
RHO_FLOOR = 1e-3
SEED = 7

RUN_MULTI_SEED = False
MULTI_SEEDS = [0, 1, 2, 3, 4]

PALETTE = {
    "adam": "#E3120B",
    "rho_adam": "#006BA2",
    "dark": "#3D3D3D",
    "mid": "#767676",
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)



## 2. Model and data

We keep the same small MLP and digits dataset as Tutorial 0. That way, the reader
can attribute the behavior change to the optimizer rather than to a new model family.


In [ ]:

class MLP(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.fc1 = nn.Linear(64, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, 10)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


def make_model():
    return MLP().to(DEVICE)


def build_digits_loaders(seed: int, batch_size: int = 128):
    digits = load_digits()
    X = digits.data.astype(np.float32) / 16.0
    y = digits.target.astype(np.int64)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
    return train_loader, test_loader


train_loader, test_loader = build_digits_loaders(SEED, BATCH_SIZE)
len(train_loader), len(test_loader)



## 3. Direct comparison: Adam versus rho-scaled Adam

Plain Adam uses its moving-average update and applies the chosen base learning rate:

```python
opt = torch.optim.Adam(model.parameters(), lr=base_lr)
loss.backward()
opt.step()
```

`RhoScaledAdam` changes only one part of this story: before the parameter update,
it computes a scalar $\rho$ from the current logits and rescales Adam's global
learning-rate factor.

```python
opt = RhoScaledAdam(model.parameters(), lr=base_lr, rhoCap=1.0)
loss.backward()
opt.step(logits=logits)
```

Internally, the optimizer does this:

```python
rho_batch = computeRho(logits, method="minmax")
scale = max(min(rho_batch, rhoCap), rhoFloor)
eff_lr = base_lr * scale
```

So this notebook is not using the exact JVP-based $	au \le \rho$ cap from Tutorial 0.
Instead, it shows the Adam version shipped in the library: the optimizer keeps Adam's
direction logic but shrinks its global step scale when the current logits imply a small
radius.


In [ ]:

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            total_loss += float(loss.item()) * len(xb)
            total_correct += int((logits.argmax(dim=1) == yb).sum())
            total_count += len(xb)
    return total_loss / total_count, total_correct / total_count


def clone_params(model):
    return [p.detach().clone() for p in model.parameters() if p.requires_grad]


def step_norm(before, after):
    total = 0.0
    for p0, p1 in zip(before, after):
        delta = (p1 - p0).reshape(-1)
        total += float(torch.dot(delta, delta))
    return total ** 0.5


def run_training(mode: str, base_lr: float, seed: int):
    set_seed(seed)
    train_loader, test_loader = build_digits_loaders(seed, BATCH_SIZE)
    model = make_model()
    if mode == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=base_lr)
    else:
        opt = RhoScaledAdam(
            model.parameters(),
            lr=base_lr,
            rhoCap=RHO_CAP,
            rhoFloor=RHO_FLOOR,
        )

    history = {
        "train_loss": [],
        "test_acc": [],
        "mean_rho": [],
        "max_r": [],
        "mean_eff_lr": [],
    }

    for _ in range(EPOCHS):
        model.train()
        batch_losses = []
        batch_rho = []
        batch_r = []
        batch_eff_lr = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            before = clone_params(model)

            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()

            rho_batch = max(float(computeRho(logits.detach(), method="minmax")), 1e-8)
            if mode == "adam":
                opt.step()
                eff_lr = float(base_lr)
            else:
                opt.step(logits=logits.detach())
                eff_lr = float(opt.getStats()["effectiveLR"])

            after = [p.detach() for p in model.parameters() if p.requires_grad]
            tau = step_norm(before, after)
            r = tau / rho_batch

            batch_losses.append(float(loss.item()))
            batch_rho.append(rho_batch)
            batch_r.append(r)
            batch_eff_lr.append(eff_lr)

        _, test_acc = evaluate(model, test_loader)
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["test_acc"].append(float(test_acc))
        history["mean_rho"].append(float(np.mean(batch_rho)))
        history["max_r"].append(float(np.max(batch_r)))
        history["mean_eff_lr"].append(float(np.mean(batch_eff_lr)))

    return history


RUN_SPECS = [
    {"key": "adam_low", "mode": "adam", "base_lr": LOW_LR, "label": f"fixed Adam (LR={LOW_LR})"},
    {"key": "adam_mid", "mode": "adam", "base_lr": MID_LR, "label": f"fixed Adam (LR={MID_LR})"},
    {"key": "adam_high", "mode": "adam", "base_lr": HIGH_LR, "label": f"fixed Adam (LR={HIGH_LR})"},
    {"key": "rho_adam_low", "mode": "rho_adam", "base_lr": LOW_LR, "label": f"rho-Adam (LR={LOW_LR})"},
    {"key": "rho_adam_mid", "mode": "rho_adam", "base_lr": MID_LR, "label": f"rho-Adam (LR={MID_LR})"},
    {"key": "rho_adam_high", "mode": "rho_adam", "base_lr": HIGH_LR, "label": f"rho-Adam (LR={HIGH_LR})"},
]


def final_summary_row(spec, hist):
    return {
        "run": spec["label"],
        "final_acc": hist["test_acc"][-1],
        "peak_r": max(hist["max_r"]),
        "final_mean_rho": hist["mean_rho"][-1],
        "final_mean_eff_lr": hist["mean_eff_lr"][-1],
    }



## 4. Single-seed run

We now run the six curves on the same split and the same model.

The comparison to watch is simple:
- fixed Adam keeps its chosen base learning rate throughout,
- rho-Adam uses the same base learning rate but shrinks the global scale when $\rho$ gets small.


In [ ]:

single_seed = {}
for spec in RUN_SPECS:
    single_seed[spec["key"]] = run_training(spec["mode"], spec["base_lr"], SEED)

summary_rows = [final_summary_row(spec, single_seed[spec["key"]]) for spec in RUN_SPECS]
pd.DataFrame(summary_rows)



## 5. First look: loss and accuracy

Start with the same two outputs as the SGD tutorial:
- training loss on a log scale,
- test accuracy on a linear scale.

Reading key:
- **color = optimizer mode**,
- **line style = learning-rate choice**,
- **markers help when curves overlap**.

The main question is whether rho-scaling changes Adam's optimization trajectory in a visible way.


In [ ]:

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

epochs = np.arange(1, EPOCHS + 1)
style_map = {
    "adam_low": {"color": PALETTE["adam"], "ls": "-", "marker": "o", "alpha": 0.82},
    "adam_mid": {"color": PALETTE["adam"], "ls": "--", "marker": "o", "alpha": 0.82},
    "adam_high": {"color": PALETTE["adam"], "ls": "-.", "marker": "o", "alpha": 0.82},
    "rho_adam_low": {"color": PALETTE["rho_adam"], "ls": "-", "marker": "s", "alpha": 0.82},
    "rho_adam_mid": {"color": PALETTE["rho_adam"], "ls": "--", "marker": "s", "alpha": 0.82},
    "rho_adam_high": {"color": PALETTE["rho_adam"], "ls": "-.", "marker": "s", "alpha": 0.82},
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True)

for spec in RUN_SPECS:
    key = spec["key"]
    hist = single_seed[key]
    style = style_map[key]
    axes[0].semilogy(
        epochs, hist["train_loss"], label=spec["label"],
        color=style["color"], ls=style["ls"], lw=2.2,
        marker=style["marker"], ms=4.5, markevery=2, alpha=style["alpha"]
    )
    axes[1].plot(
        epochs, hist["test_acc"], label=spec["label"],
        color=style["color"], ls=style["ls"], lw=2.2,
        marker=style["marker"], ms=4.5, markevery=2, alpha=style["alpha"]
    )

axes[0].set_title("Training loss")
axes[1].set_title("Test accuracy")
axes[0].set_ylabel("loss")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0.0, 1.02)

for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.25)

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
fig.suptitle("Fixed Adam versus rho-scaled Adam", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()



## 6. Diagnostic view: effective global learning-rate scale and normalized step size

Now look at the controller quantities directly.

The left panel shows the optimizer's global learning-rate scale. For fixed Adam this
is just the chosen base learning rate. For rho-Adam it shrinks whenever the observed
logits imply a smaller radius.

The right panel shows the **actual** normalized step size $r = 	au / \rho$ measured from
the parameter update norm. Unlike Tutorial 0, rho-Adam does not impose the exact
hard cap $	au \le \rho$; it rescales Adam's base factor, so this panel is a diagnostic,
not a hard safety guarantee.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True)

for spec in RUN_SPECS:
    key = spec["key"]
    hist = single_seed[key]
    style = style_map[key]
    axes[0].semilogy(
        epochs, hist["mean_eff_lr"], label=spec["label"],
        color=style["color"], ls=style["ls"], lw=2.1,
        marker=style["marker"], ms=4.0, markevery=2, alpha=0.72
    )
    axes[1].semilogy(
        epochs, hist["max_r"], label=spec["label"],
        color=style["color"], ls=style["ls"], lw=2.1,
        marker=style["marker"], ms=4.0, markevery=2, alpha=0.72
    )

axes[0].set_title("Effective global LR scale")
axes[1].set_title("Normalized step size (max r per epoch)")
axes[0].set_ylabel("effective lr")
axes[1].set_ylabel("max r")
axes[1].axhline(1.0, color=PALETTE["dark"], ls=":", lw=1.2)

for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.25)

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
fig.suptitle("Adam diagnostics", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()



## 7. Summary table

The figure gives the visual story. The table below summarizes the same run numerically.


In [ ]:

summary_df = pd.DataFrame(summary_rows)
summary_df["final_acc"] = summary_df["final_acc"].map(lambda x: round(float(x), 3))
summary_df["peak_r"] = summary_df["peak_r"].map(lambda x: round(float(x), 3))
summary_df["final_mean_rho"] = summary_df["final_mean_rho"].map(lambda x: round(float(x), 4))
summary_df["final_mean_eff_lr"] = summary_df["final_mean_eff_lr"].map(lambda x: round(float(x), 6))
summary_df



## 8. Optional: multiseed check

The single-seed run is the fastest way to understand the mechanism. If you want a more stable summary,
flip `RUN_MULTI_SEED = True` and rerun the next cell.


In [ ]:

if RUN_MULTI_SEED:
    multi_rows = []
    for seed in MULTI_SEEDS:
        for spec in RUN_SPECS:
            hist = run_training(spec["mode"], spec["base_lr"], seed)
            multi_rows.append({
                "run": spec["label"],
                "seed": seed,
                "final_acc": hist["test_acc"][-1],
                "peak_r": max(hist["max_r"]),
            })

    multi_df = pd.DataFrame(multi_rows)
    display(
        multi_df.groupby("run", as_index=False)
        .agg(final_acc_mean=("final_acc", "mean"), final_acc_std=("final_acc", "std"), peak_r_mean=("peak_r", "mean"))
        .round(3)
    )
else:
    print("Set RUN_MULTI_SEED = True to execute the multiseed check.")



## 9. Where to go next

- Return to [`tutorials/00_step_controller_intro.ipynb`](tutorials/00_step_controller_intro.ipynb) if you want the exact JVP-based cap story.
- Use the figure notebooks in [`notebooks/`](notebooks) for lightweight paper reproductions.
- The next natural extension is the same comparison for SGD with momentum.
